# Preparación de datos para el modelado

In [1]:
%load_ext watermark
%watermark

Last updated: 2026-11-19T19:43:34.273324-05:00

Python implementation: CPython
Python version       : 3.12.16
IPython version      : 9.17.1

Compiler    : Clang 20.1.8 
OS          : Darwin
Release     : 23.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 11
Architecture: 64bit



In [1]:
import pandas as pd
%matplotlib inline 
# no crea widget ni nada para no presentar resultados.
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
import matplotlib as plt
import seaborn as sns
from matplotlib.pyplot import figure
import warnings
warnings.filterwarnings("ignore")


plt.rcParams['figure.figsize'] = (10,10) # 10 pulgadas de ancho y 10 de alto para todos los plots

In [2]:
bank_dp = pd.read_parquet("../output/data-bank_3_EDA.parquet",engine='pyarrow')

In [3]:
bank_dp.shape

(11152, 18)

In [4]:
bank_dp.columns

Index(['edad', 'trabajo', 'estadoCivil', 'educacion', 'flag_mora',
       'prom_saldoAnual', 'flag_hipotecario', 'flag_prestamo', 'tipo_contacto',
       'dia_UltContacto', 'mes_UltContacto', 'seg_UltContacto',
       'cant_contactoCampanha', 'cant_contactoBeforeCampanha',
       'result_campanha', 'FlagTarget_DP', 'Rango_edad', 'duration_status'],
      dtype='object')

## Estrategia de limpieza de datos

Dado que las variables categóricas dominan en el conjunto de datos y la cantidad de variables numéricas débilmente correlacionadas no es más de 4, necesitamos transformar las variables categóricas para aumentar la capacidad del modelo para generalizar datos. (no podemos soltarlos

Se debe prestar especial atención a los valores categóricos y a la variable de duración `seg_UltContacto`, esta por su fuerte correlación con la variable objetivo, que pueden tratarse como binarias. Se sugiere usar agrupamiento y transformación simple en consecuencia (0 y 1). Para categorías de más de 3 tipos de opción posible (matrimonial y educación) se propone usar la orientación codificada: permitirá relacionar correctamente los valores con la variable objetivo y usar categorías indicadas en forma numérica.

En algunos casos, se propone reescalar para normalizar los datos

In [5]:
bank_dp2 = bank_dp.copy()

In [6]:
# Replacing values with binary ()
bank_dp2.tipo_contacto = bank_dp2.tipo_contacto.map({'cellular': 1, 'telephone': 0,'unknown': 2}).astype('uint16') 
bank_dp2.flag_prestamo = bank_dp2.flag_prestamo.map({'yes': 1, 'unknown': 0, 'no' : 0}).astype('uint8')
bank_dp2.flag_hipotecario = bank_dp2.flag_hipotecario.map({'yes': 1, 'unknown': 0, 'no' : 0}).astype('uint8')
bank_dp2.flag_mora = bank_dp2.flag_mora.map({'no': 1, 'unknown': 0, 'yes': 0}).astype('uint8')
 # binary has contact or not
bank_dp2['flag_ContactBeforeCamp'] = bank_dp2.cant_contactoBeforeCampanha.apply(lambda x: 1 if x > 0 else 0).astype('uint8')

# binary if were was an outcome of marketing campane
bank_dp2.result_campanha = bank_dp2.result_campanha.map({'other':0,'unknown':0, 'failure':0, 'success':1}).astype('uint8') 

In [8]:
bank_dp2.head(5)

,edad,trabajo,estadoCivil,educacion,flag_mora,prom_saldoAnual,flag_hipotecario,flag_prestamo,tipo_contacto,dia_UltContacto,mes_UltContacto,seg_UltContacto,cant_contactoCampanha,cant_contactoBeforeCampanha,result_campanha,FlagTarget_DP,Rango_edad,duration_status,flag_ContactBeforeCamp
0,59,admin.,married,secondary,1,2343,1,0,2,5,may,1042,1,0,0,1,55-60,encima_prom,0
1,56,admin.,married,secondary,1,45,0,0,2,5,may,1467,1,0,0,1,55-60,encima_prom,0
2,41,technician,married,secondary,1,1270,1,0,2,5,may,1389,1,0,0,1,41-45,encima_prom,0
3,55,services,married,secondary,1,2476,1,0,2,5,may,579,1,0,0,1,51-55,encima_prom,0
4,54,admin.,married,tertiary,1,184,0,0,2,5,may,673,2,0,0,1,51-55,encima_prom,0


In [7]:
bank_dp2.seg_UltContacto.unique()

array([1042, 1467, 1389, ..., 1504,  818, 1812], shape=(1428,))

#### Técnicas de codificación
Para los algoritmos de aprendizaje automático, los datos categóricos pueden ser extremadamente útiles. Sin embargo, en su forma original, es irreconocible para la mayoría de los modelos. Para resolver este problema, podemos utilizar diferentes técnicas de “codificación” para hacer legibles nuestros datos categóricos.

In [8]:
# fucntion to One Hot Encoding
def encodeOneHot(data, col):
    return pd.concat([data, pd.get_dummies(col, prefix=col.name)], axis=1)

# One Hot encoding of 3 variable -- Pivoteo por valor
bank_dp2 = encodeOneHot(bank_dp2, bank_dp2.trabajo)
bank_dp2 = encodeOneHot(bank_dp2, bank_dp2.mes_UltContacto)
bank_dp2 = encodeOneHot(bank_dp2, bank_dp2.estadoCivil)
bank_dp2 = encodeOneHot(bank_dp2, bank_dp2.educacion)


In [9]:
# Drop tranfromed features
bank_dp2.drop(['trabajo', 'mes_UltContacto','estadoCivil','educacion'], axis=1, inplace=True)

In [10]:
bank_dp2.columns

Index(['edad', 'flag_mora', 'prom_saldoAnual', 'flag_hipotecario',
       'flag_prestamo', 'tipo_contacto', 'dia_UltContacto', 'seg_UltContacto',
       'cant_contactoCampanha', 'cant_contactoBeforeCampanha',
       'result_campanha', 'FlagTarget_DP', 'Rango_edad', 'duration_status',
       'flag_ContactBeforeCamp', 'trabajo_admin.', 'trabajo_blue-collar',
       'trabajo_entrepreneur', 'trabajo_housemaid', 'trabajo_management',
       'trabajo_retired', 'trabajo_self-employed', 'trabajo_services',
       'trabajo_student', 'trabajo_technician', 'trabajo_unemployed',
       'trabajo_unknown', 'mes_UltContacto_apr', 'mes_UltContacto_aug',
       'mes_UltContacto_dec', 'mes_UltContacto_feb', 'mes_UltContacto_jan',
       'mes_UltContacto_jul', 'mes_UltContacto_jun', 'mes_UltContacto_mar',
       'mes_UltContacto_may', 'mes_UltContacto_nov', 'mes_UltContacto_oct',
       'mes_UltContacto_sep', 'estadoCivil_divorced', 'estadoCivil_married',
       'estadoCivil_single', 'educacion_primary',

In [11]:
[c for c in bank_dp2.columns if c.startswith('dia_UltContacto')]

['dia_UltContacto']

In [12]:
'''Drop the dublicates'''
bank_dp2.drop_duplicates(inplace=True) 

In [13]:
bank_dp2.seg_UltContacto.unique()

array([1042, 1467, 1389, ..., 1504,  818, 1812], shape=(1428,))

In [14]:
'''Convert Duration Call into 5 category'''
def Categ_duration(data):
    data.loc[data['seg_UltContacto'] <= 102, 'duracion'] = 1
    data.loc[(data['seg_UltContacto'] > 102) & (data['seg_UltContacto'] <= 180)  , 'duracion'] = 2
    data.loc[(data['seg_UltContacto'] > 180) & (data['seg_UltContacto'] <= 319)  , 'duracion'] = 3
    data.loc[(data['seg_UltContacto'] > 319) & (data['seg_UltContacto'] <= 645), 'duracion'] = 4
    data.loc[data['seg_UltContacto']  > 645, 'duracion'] = 5
    return data
Categ_duration(bank_dp2);

In [15]:
# eliminar la columna
bank_dp2.drop(['seg_UltContacto'], axis=1, inplace=True)

In [16]:
#No sale en doc
bank_dp2.duracion.unique()

array([5., 4., 3., 1., 2.])

In [19]:
# instalar esta libreria en cmd:  pip install category_encoders
#import category_encoders as ce

In [17]:
''' Target encoding for two categorical feature '''
# save target variable before transformation
FlagTarget_DP = bank_dp2.FlagTarget_DP
# Create target encoder object and transoform two value xq??
#target_encode = ce.target_encoder.TargetEncoder(cols=['estadoCivil', 'educacion']).fit(bank_dp2, FlagTarget_DP)
#numeric_dataset = target_encode.transform(bank_dp2)
# drop target variable
#numeric_dataset.drop('y', axis=1, inplace=True)

In [19]:
print("Porcentaje de la variable objetivo perteneciente a la clase Sí",(FlagTarget_DP.value_counts()[1]/len(FlagTarget_DP))*100)
print("Porcentaje de la variable objetivo perteneciente a la clase No",(FlagTarget_DP.value_counts()[0]/len(FlagTarget_DP))*100)

Porcentaje de la variable objetivo perteneciente a la clase Sí 47.40853658536585
Porcentaje de la variable objetivo perteneciente a la clase No 52.59146341463414


In [20]:
bank_dp2.info(verbose=True)
#bank_dp2.dtypes

<class 'pandas.core.frame.DataFrame'>
Index: 11152 entries, 0 to 11161
Data columns (total 46 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   edad                         11152 non-null  int64  
 1   flag_mora                    11152 non-null  uint8  
 2   prom_saldoAnual              11152 non-null  int64  
 3   flag_hipotecario             11152 non-null  uint8  
 4   flag_prestamo                11152 non-null  uint8  
 5   tipo_contacto                11152 non-null  uint16 
 6   dia_UltContacto              11152 non-null  int64  
 7   cant_contactoCampanha        11152 non-null  int64  
 8   cant_contactoBeforeCampanha  11152 non-null  int64  
 9   result_campanha              11152 non-null  uint8  
 10  FlagTarget_DP                11152 non-null  uint8  
 11  Rango_edad                   11152 non-null  object 
 12  duration_status              11152 non-null  object 
 13  flag_ContactBeforeCam

Remover del data set los valores categóricos que quedaron y que fueron utilizados para la etapa de análisis de datos, ahora entraremos a la etapa del modelado.

In [21]:
# Nota: 'duration_status' se crea en el notebook 3 (EDA). Si no está presente,
# se eliminan solo las columnas auxiliares que existan para evitar KeyError.
cols_drop = [c for c in ['Rango_edad','duration_status'] if c in bank_dp2.columns]
bank_dp2 = bank_dp2.drop(columns=cols_drop)
print('Auxiliares eliminadas:', cols_drop)

Auxiliares eliminadas: ['Rango_edad', 'duration_status']


In [24]:
#bank_dp2.info(verbose=True)

In [22]:
#EXPORT PARQUET
path="../output/"
bank_dp2.to_parquet(path+"data-bank_4_PreModeling.parquet",engine="pyarrow")

In [23]:
# Verificación final del dataset pre-modelado
print('Shape final:', bank_dp2.shape)
print('Columnas de texto restantes:', [c for c in bank_dp2.columns if bank_dp2[c].dtype=='object'])
print('Distribución del target:')
print(bank_dp2['FlagTarget_DP'].value_counts(normalize=True).round(4))

Shape final: (11152, 44)
Columnas de texto restantes: []
Distribución del target:
FlagTarget_DP
0    0.5259
1    0.4741
Name: proportion, dtype: float64
